In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.1 MB/s eta 0:00:00


In [3]:
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# Load the model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

index = faiss.read_index("/content/drive/MyDrive/squad_index.faiss")
with open("/content/drive/MyDrive/squad_chunks.pkl", "rb") as f:
    final_chunks = pickle.load(f)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

def rechunk_by_sentences(chunks, sentences_per_chunk=5, overlap=2):
    # Step 1 — Join all chunks into full text
    full_text = " ".join(chunks)

    # Step 2 — Split into complete sentences
    sentences = sent_tokenize(full_text)
    print(f"Total sentences found: {len(sentences)}")

    # Step 3 — Regroup into overlapping chunks
    new_chunks = []
    i = 0
    while i < len(sentences):
        chunk = " ".join(sentences[i:i+sentences_per_chunk])
        new_chunks.append(chunk)
        i += (sentences_per_chunk - overlap)

    print(f"Old chunks: {len(chunks)} → New chunks: {len(new_chunks)}")
    return new_chunks

# Apply fix
final_chunks = rechunk_by_sentences(final_chunks)


Total sentences found: 108512
Old chunks: 43049 → New chunks: 36171


Explanation : this code is for rechunking.Basically, it takes chunks, no of sentence per chunk and overlap as input.
It joins all chuks, uses nltk to split sentence wise and store in sentence. later each chunk will have no of sentence= (overlap + extra) like n=5 then 2 overlapped 3 new one

In [6]:
import numpy as np
def deduplicate_chunks(chunks, min_words=20):
    """Remove repeated sentences and filter short garbage chunks."""
    cleaned_chunks = []
    seen_sentences = set()

    for chunk in chunks:
        sentences = sent_tokenize(chunk)
        unique_sentences = []

        for sent in sentences:
            sent_clean = sent.strip()
            if sent_clean not in seen_sentences and len(sent_clean.split()) > 5:
                seen_sentences.add(sent_clean)
                unique_sentences.append(sent_clean)

        if unique_sentences:
            cleaned = " ".join(unique_sentences)
            # Filter out chunks that are too short
            if len(cleaned.split()) >= min_words:
                cleaned_chunks.append(cleaned)

    print(f"Chunks after dedup + filter: {len(cleaned_chunks)}")
    return cleaned_chunks

# Re-apply
final_chunks = deduplicate_chunks(final_chunks, min_words=20)

# Rebuild index
chunk_embeddings = embed_model.encode(final_chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print(f"✅ Index rebuilt with {index.ntotal} chunks!")

Chunks after dedup + filter: 36035


Batches:   0%|          | 0/1127 [00:00<?, ?it/s]

✅ Index rebuilt with 36035 chunks!


This works in corpus level. Removes redundancy,globally.Also remove chunks less than 20 words and rebuild faiss indes

In [7]:
def get_context(query, k=3):
    """
    Member 1's Retrieval Function.
    Input: User Question (string)
    Output: List of relevant text chunks
    """
    query_vec = embed_model.encode([query])
    distances, indices = index.search(query_vec, k)
    return [final_chunks[i] for i in indices[0]]

# Example for Member 2 to test:
# context = get_context("What is the capital of France?")

Retrives top k most relevant chunks

module 2

In [40]:
!pip install groq
from dotenv import load_dotenv
import os

load_dotenv('.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
print("✅ API key loaded!")

✅ API key loaded!


In [30]:
def generate_answer(query, context_chunks):
    combined_context = "\n\n".join(context_chunks)

    words = combined_context.split()
    if len(words) > 300:
        combined_context = " ".join(words[:300])

    prompt = f"""You are a question answering assistant.
Use ONLY the context below to answer the question.
Answer the question DIRECTLY — do not mention experiments on animals or historical background unless specifically asked.
Write a COMPLETE, FULL sentence of at least 10 words.
If the context does not directly answer the question, say "The context does not contain a direct answer."

Context:
{combined_context}

Question: {query}

Direct answer:"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": "You answer questions directly and concisely using only provided context. Never mention animal experiments when asked about human medical conditions unless specifically asked."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=200,
            temperature=0.1  # lower temperature = more focused
        )

        answer = response.choices[0].message.content.strip()

        if len(answer.split()) < 5:
            return "The context does not contain enough information to answer this question."

        return answer

    except Exception as e:
        print(f"  ⚠️ Generation error: {e}")
        return "Answer generation failed."

here answer is generated based on strict prompt. Also, if length is less than 5 it shd give a fallback.Also it generates ansfrom the context.

In [12]:
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

# For semantic similarity
#embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # same as Member 1

# For NLI (entailment checking)
# Replace your old nli_pipeline with this
nli_pipeline = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    top_k=None  # returns all 3 labels reliably
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

NLI - Natural language interface. It logically supports answer. This model gives 3 lables : entailment(context supports answer),contradiction(context say opposite),neutral(context doesnt clearly support)

In [13]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def split_sentences(text):
    return sent_tokenize(text)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


for sentence wise checking if hallucination has happened

In [14]:
def semantic_similarity_score(answer, context_chunks):
    combined_context = " ".join(context_chunks)
    ans_emb = embed_model.encode(answer, convert_to_tensor=True)
    ctx_emb = embed_model.encode(combined_context, convert_to_tensor=True)
    score = util.cos_sim(ans_emb, ctx_emb).item()
    return score  # 0 to 1

cosine similarity

In [28]:
def is_context_relevant(query, context_chunks, threshold=0.30):
    """
    Use NLI entailment to check if context is relevant to query.
    If context entails/relates to query → relevant.
    """
    best_entail = 0
    best_neutral = 0

    for chunk in context_chunks:
        results = nli_pipeline(
            f"{chunk} [SEP] {query}",
            truncation=True,
            top_k=None,
            max_length=512
        )
        if isinstance(results[0], list):
            results = results[0]

        scores = {r['label'].lower(): r['score'] for r in results}
        entail = scores.get('entailment', 0)
        neutral = scores.get('neutral', 0)

        if entail > best_entail:
            best_entail = entail
        if neutral > best_neutral:
            best_neutral = neutral

    # Relevant if context either entails or is neutral (related) to query
    # Irrelevant if context only contradicts query (completely different topic)
    relevance_score = best_entail + (best_neutral * 0.5)
    print(f"  NLI relevance score: {relevance_score:.2f}")
    return relevance_score > threshold, relevance_score

Checks if retrived chunks is relevant to user query.
relevance_score = best_entail + (best_neutral * 0.5)

In [16]:
def check_entailment(sentence, context_chunks):
    sentence_emb = embed_model.encode(sentence, convert_to_tensor=True)

    best_sim = 0
    best_nli = "neutral"
    best_nli_scores = {}

    for chunk in context_chunks:
        chunk_emb = embed_model.encode(chunk, convert_to_tensor=True)
        sim = util.cos_sim(sentence_emb, chunk_emb).item()
        if sim > best_sim:
            best_sim = sim

        results = nli_pipeline(
            f"{chunk} [SEP] {sentence}",
            truncation=True,
            top_k=None,
            max_length=512
        )
        if isinstance(results[0], list):
            results = results[0]

        scores = {r['label'].lower(): r['score'] for r in results}
        entail = scores.get('entailment', 0)
        neutral = scores.get('neutral', 0)
        contra  = scores.get('contradiction', 0)

        if entail > neutral and entail > contra:
            best_nli = 'entailment'
            best_nli_scores = scores
        elif contra > neutral and contra > entail:
            if best_nli != 'entailment':
                best_nli = 'contradiction'
                best_nli_scores = scores

    print(f"  Semantic sim: {best_sim:.2f} | NLI: {best_nli}")

    # Short answer boost — if answer words appear directly in context
    answer_words = set(sentence.lower().split())
    for chunk in context_chunks:
        chunk_words = set(chunk.lower().split())
        overlap = answer_words & chunk_words
        overlap_ratio = len(overlap) / len(answer_words) if answer_words else 0
        if overlap_ratio > 0.7:  # 70% of answer words found in context
            print(f"  Word overlap boost: {overlap_ratio:.2f} → ENTAILMENT")
            return 'entailment', max(best_sim, 0.65)

    # Normal decision logic
    if best_sim >= 0.64:
        return 'entailment', best_sim
    elif best_sim > 0.50:
        if best_nli == 'contradiction':
            return 'contradiction', best_sim
        return 'entailment', best_sim
    else:
        if best_nli == 'entailment':
            return 'neutral', best_sim
        return 'contradiction', best_sim


Core verification function — checks each answer sentence.
Two layers:
  Layer 1: Semantic similarity (embedding cosine sim)
  Layer 2: NLI entailment check
  Layer 3: Word overlap boost for short answers

Decision logic:
  sim >= 0.64         → ENTAILMENT (supported)
  sim 0.50-0.63
    + NLI=contra      → CONTRADICTION (hallucinated)
    + NLI=entail      → ENTAILMENT (supported)
  sim < 0.50          → CONTRADICTION (hallucinated)

In [17]:
def compute_hallucination_score(answer, context_chunks):
    sentences = split_sentences(answer)
    total_score = 0
    sentence_results = []

    for sent in sentences:
        label, score = check_entailment(sent, context_chunks)
        sentence_results.append({"sentence": sent, "label": label, "score": round(score, 2)})

        if label == "entailment":
            total_score += 1.0
        elif label == "neutral":
            # High confidence neutral = model unsure = treat as partial hallucination
            if score > 0.80:
                total_score += 0.2   # ← penalize heavily
            else:
                total_score += 0.5
        else:  # contradiction
            total_score += 0.0

    support_ratio = total_score / len(sentences) if sentences else 0
    sem_score = semantic_similarity_score(answer, context_chunks)
    hallucination_score = 1 - (0.6 * support_ratio + 0.4 * sem_score)

    return round(hallucination_score, 2), sentence_results

hallucination = 1 - (0.6 × support_ratio + 0.4 × sem_score)
  Higher score = more hallucinated

In [20]:
def classify_hallucination(score):
    if score < 0.3:
        return "✅ Supported"
    elif score < 0.65:
        return "⚠️ Partially Supported"
    else:
        return "🔴 Hallucinated"

In [21]:
import re

def clean_text(text):
    # Add space after period if missing (e.g., "child.It" → "child. It")
    text = re.sub(r'\.(?=[A-Z])', '. ', text)
    # Fix multiple spaces
    text = re.sub(r' +', ' ', text)
    return text.strip()

In [34]:
def check_answer_relevance(query, answer, threshold=0.20):
    """
    Check if answer is relevant to query using
    semantic similarity instead of NLI.
    """
    query_emb = embed_model.encode(query, convert_to_tensor=True)
    answer_emb = embed_model.encode(answer, convert_to_tensor=True)

    similarity = util.cos_sim(query_emb, answer_emb).item()
    print(f"  Answer relevance to query: {similarity:.2f}")

    return similarity > threshold, similarity

In [35]:
def hallucination_detection_module(query, answer, context_chunks):
    answer = clean_text(answer)

    # Pass answer into relevance check
    is_relevant, relevance_score = is_context_relevant(query, context_chunks)

    if not is_relevant:
        print(f"\n⚠️ WARNING: Retrieved context is NOT relevant to the query.")
        print(f"   Relevance Score: {relevance_score:.2f}")
        print(f"   Cannot verify answer — no relevant documents found.")
        return {
            "query": query,
            "answer": answer,
            "hallucination_score": None,
            "label": "⚠️ Unverifiable — No Relevant Context",
            "sentence_analysis": [],
            "pass_to_next": False
        }

    if not check_answer_relevance(query, answer):
        print(f"⚠️ Answer does not address the query!")
        return {
            "query": query,
            "answer": answer,
            "hallucination_score": 0.80,
            "label": "⚠️ Partially Supported — Answer Off-Topic",
            "sentence_analysis": [],
            "pass_to_next": False  # send to Member 3 for correction
        }

    score, sentence_details = compute_hallucination_score(answer, context_chunks)
    label = classify_hallucination(score)

    print(f"\n🔍 Query: {query}")
    print(f"📄 Answer: {answer}")
    print(f"\n📊 Hallucination Score: {score}")
    print(f"🏷️ Classification: {label}")
    print("\n📝 Sentence-level Analysis:")
    for item in sentence_details:
        print(f"  [{item['label'].upper()}] ({item['score']}) {item['sentence']}")

    return {
        "query": query,
        "answer": answer,
        "hallucination_score": score,
        "label": label,
        "sentence_analysis": sentence_details,
        "pass_to_next": score < 0.65
    }

Main validation function — connects everything.
Flow:
  1. clean_text(answer)
  2. is_context_relevant() → if irrelevant → return Unverifiable
  3. compute_hallucination_score()
  4. classify_hallucination()
  5. return result dict with pass_to_next flag for Member 3

In [36]:
def h_pipeline(query):
    """
    Complete Member 2 Pipeline

    Input:  User query (string)
    Output: Validation result dict

    Flow:
    get_context() → generate_answer() → hallucination_detection_module()
    """
    print(f"\n{'='*60}")
    print(f"📥 Query: {query}")
    print(f"{'='*60}")

    # Step 1 — Retrieve context (Member 1's function)
    print("\n🔍 Step 1: Retrieving context...")
    context = get_context(query, k=7)
    for i, chunk in enumerate(context[:2]):  # show first 2 chunks
        print(f"  Chunk {i+1}: {chunk[:100]}...")

    # Step 2 — Generate answer from context
    print("\n🤖 Step 2: Generating answer from context...")
    answer = generate_answer(query, context)
    print(f"  Generated: {answer}")

    # Step 3 — Validate the generated answer
    print("\n🔎 Step 3: Validating answer...")
    result = hallucination_detection_module(query, answer, context)

    # Step 4 — Decision for Member 3
    print(f"\n{'='*60}")
    print(f"📤 Passing to Member 3: {result['pass_to_next']}")
    if result['pass_to_next']:
        print(f"✅ Answer is reliable — deliver to user")
    else:
        print(f"⚠️ Answer needs correction before delivery")
    print(f"{'='*60}")

    return result

Complete Member 2 pipeline that connects all steps:
  Step 1: get_context() ← Member 1's function
  Step 2: generate_answer() ← Groq LLaMA
  Step 3: hallucination_detection_module()
  Step 4: prints pass_to_next decision for Member 3

In [39]:
# Test 1 — relevant query
result1 = h_pipeline("What is diabetes?")

print("\n")

# Test 2 — another relevant query
result2 = h_pipeline("Who discovered insulin?")

print("\n")

# Test 3 — irrelevant query
result3 = h_pipeline("Name a river?")
print("\n")

# Test 4 — irrelevant query
result4 = h_pipeline("Hi how are you?")



📥 Query: What is diabetes?

🔍 Step 1: Retrieving context...
  Chunk 1: In 1869, Oskar Minkowski and Joseph von Mering found that diabetes could be induced in dogs by surgi...
  Chunk 2: Increased consumption of sugar has been tied to these three, among others. Obesity levels have more ...

🤖 Step 2: Generating answer from context...
  Generated: The context does not contain a direct answer to what diabetes is, but it does mention that diabetes could be induced by surgical removal of the pancreas and that obesity and diet are high risk factors for diabetes.

🔎 Step 3: Validating answer...
  NLI relevance score: 0.43
  Answer relevance to query: 0.68
  Semantic sim: 0.63 | NLI: entailment

🔍 Query: What is diabetes?
📄 Answer: The context does not contain a direct answer to what diabetes is, but it does mention that diabetes could be induced by surgical removal of the pancreas and that obesity and diet are high risk factors for diabetes.

📊 Hallucination Score: 0.11
🏷️ Classification: ✅ 

In [38]:
# This gives Member 3 the correction case
context = get_context("What is diabetes?", k=7)
result = hallucination_detection_module(
    "What is diabetes?",
    "Diabetes is caused by eating too much sugar as a child. The pancreas stops working entirely.",
    context
)
print(result)

  NLI relevance score: 0.43
  Answer relevance to query: 0.65
  Semantic sim: 0.62 | NLI: contradiction
  Semantic sim: 0.42 | NLI: contradiction

🔍 Query: What is diabetes?
📄 Answer: Diabetes is caused by eating too much sugar as a child. The pancreas stops working entirely.

📊 Hallucination Score: 0.74
🏷️ Classification: 🔴 Hallucinated

📝 Sentence-level Analysis:
  [CONTRADICTION] (0.62) Diabetes is caused by eating too much sugar as a child.
  [CONTRADICTION] (0.42) The pancreas stops working entirely.
{'query': 'What is diabetes?', 'answer': 'Diabetes is caused by eating too much sugar as a child. The pancreas stops working entirely.', 'hallucination_score': 0.74, 'label': '🔴 Hallucinated', 'sentence_analysis': [{'sentence': 'Diabetes is caused by eating too much sugar as a child.', 'label': 'contradiction', 'score': 0.62}, {'sentence': 'The pancreas stops working entirely.', 'label': 'contradiction', 'score': 0.42}], 'pass_to_next': False}


## DONT CHECK ANY CODE BELOW THIS!!!!!!

tries tries

In [ ]:
test_cases = [
    # (query, answer, expected_label)
    ("What is diabetes?",
     "Diabetes is caused by absence of a substance produced by the pancreas.",
     "✅ Supported"),

    ("Who discovered insulin?",
     "Insulin was discovered by Frederick Banting and Charles Best in 1921.",
     "✅ Supported"),

    ("What is diabetes?",
     "Diabetes is caused by eating too much sugar as a child.",
     "🔴 Hallucinated"),

    ("How does insulin work?",
     "Insulin is produced by the liver and increases blood pressure.",
     "⚠️ Partially Supported"),

    ("Give me information about giraffes",
     "Giraffes are very tall animals that live in Africa.",
     "⚠️ Unverifiable"),

    ("What is the capital of France?",
     "The capital of France is Paris.",
     "⚠️ Unverifiable"),
]

print("="*60)
for query, answer, expected in test_cases:
    ctx = get_context(query, k=7)
    result = hallucination_detection_module(query, answer, ctx)
    status = "✅ PASS" if expected in result['label'] else "❌ FAIL"
    print(f"\n{status}")
    print(f"Expected : {expected}")
    print(f"Got      : {result['label']}")
    print(f"Score    : {result['hallucination_score']}")
    print("="*60)

  NLI relevance score: 0.52
  Semantic sim: 0.64 | NLI: entailment

🔍 Query: What is diabetes?
📄 Answer: Diabetes is caused by absence of a substance produced by the pancreas.

📊 Hallucination Score: 0.12
🏷️ Classification: ✅ Supported

📝 Sentence-level Analysis:
  [ENTAILMENT] (0.64) Diabetes is caused by absence of a substance produced by the pancreas.

✅ PASS
Expected : ✅ Supported
Got      : ✅ Supported
Score    : 0.12
  NLI relevance score: 0.54
  Semantic sim: 0.64 | NLI: contradiction

🔍 Query: Who discovered insulin?
📄 Answer: Insulin was discovered by Frederick Banting and Charles Best in 1921.

📊 Hallucination Score: 0.13
🏷️ Classification: ✅ Supported

📝 Sentence-level Analysis:
  [ENTAILMENT] (0.64) Insulin was discovered by Frederick Banting and Charles Best in 1921.

✅ PASS
Expected : ✅ Supported
Got      : ✅ Supported
Score    : 0.13
  NLI relevance score: 0.52
  Semantic sim: 0.59 | NLI: contradiction

🔍 Query: What is diabetes?
📄 Answer: Diabetes is caused by eating to

In [ ]:
query = "Give me information about giraffes"
context = get_context(query,k=5)
answer = "Giraffes are very tall.They live in water.They like biriyani.They are vegetarians"
answer = clean_text(answer)
result = hallucination_detection_module(query, answer, context)


  Context relevance score: 0.41
  NLI best scores → {'contradiction': 0.728837251663208, 'neutral': 0.2349974662065506, 'entailment': 0.036165278404951096} → contradiction
  NLI best scores → {'contradiction': 0.996864378452301, 'neutral': 0.002746081678196788, 'entailment': 0.00038958716322667897} → contradiction
  NLI best scores → {'contradiction': 0.9427000880241394, 'neutral': 0.05552225932478905, 'entailment': 0.0017776743043214083} → contradiction
  NLI best scores → {'contradiction': 0.9988521337509155, 'neutral': 0.0009425688185729086, 'entailment': 0.00020530048641376197} → contradiction

🔍 Query: Give me information about giraffes
📄 Answer: Giraffes are very tall. They live in water. They like biriyani. They are vegetarians

📊 Hallucination Score: 0.83
🏷️ Classification: 🔴 Hallucinated

📝 Sentence-level Analysis:
  [CONTRADICTION] (0.73) Giraffes are very tall.
  [CONTRADICTION] (1.0) They live in water.
  [CONTRADICTION] (0.94) They like biriyani.
  [CONTRADICTION] (1.0) Th

trial

In [ ]:
# Run this debug cell to see raw output
chunk = context[0]  # first retrieved chunk
test_sentence = "It is caused by eating too much sugar as a child."

raw = nli_pipeline(f"{chunk} [SEP] {test_sentence}", truncation=True)
print(raw)

[[{'label': 'contradiction', 'score': 0.683238685131073}, {'label': 'neutral', 'score': 0.316385954618454}, {'label': 'entailment', 'score': 0.00037529238034039736}]]


In [66]:
context = get_context("Give me information about giraffes")
for i, chunk in enumerate(context):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
They have been bred for herding livestock, hunting (e.g. pointers and hounds), rodent control, guarding, helping fishermen with nets, detection dogs, and pulling loads, in addition to their roles as companions.

--- Chunk 2 ---
Visitors can see about 243 specimens of different species including kangaroos, giant panda, gorillas, caracal, hyena, hippos, jaguar, giraffe, lemur, lion, among others.

--- Chunk 3 ---
Its mosaic floor depicts King David as Orpheus, identified by his name in Hebrew letters. Near him were lion cubs, a giraffe and a snake listening to him playing a lyre. A further portion of the floor was divided by medallions formed by vine leaves, each of which contains an animal: a lioness suckling her cub, a giraffe, peacocks, panthers, suckling her cub, a giraffe, peacocks, panthers, bears, a zebra and so on.


In [65]:
label, score = keyword_fact_check("It is caused by eating too much sugar as a child.")
print(label, score)
# Should print: contradiction 0.95

NameError: name 'keyword_fact_check' is not defined

one full

In [ ]:
# ============================================================
# COMPLETE HALLUCINATION DETECTION MODULE — RUN THIS ONE CELL
# ============================================================

import re
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

# --- Models ---
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
nli_pipeline = pipeline("text-classification", model="facebook/bart-large-mnli", top_k=None)

# --- Known Facts Database ---
KNOWN_FACTS = [
    ("eating too much sugar causes diabetes", "contradiction"),
    ("eating sugar causes diabetes", "contradiction"),
    ("sugar causes diabetes", "contradiction"),
    ("caused by eating too much sugar", "contradiction"),
    ("pancreas stops working entirely", "contradiction"),
    ("avoid all carbohydrates", "contradiction"),
    ("doesnt spike insulin", "contradiction"),
    ("doesn't spike insulin", "contradiction"),
    ("does not spike insulin", "contradiction"),
    ("insulin is not produced", "contradiction"),
    ("not related to blood sugar", "contradiction"),
    ("doesnt spikes insulin", "contradiction"),      # ← typo version
    ("doesnt spike insulin", "contradiction"),
]

# --- Text Cleaner ---
def clean_text(text):
    text = re.sub(r'\.(?=[A-Z])', '. ', text)
    text = re.sub(r' +', ' ', text)
    return text.strip()

# --- Sentence Splitter ---
def split_sentences(text):
    return sent_tokenize(text)

# --- Keyword Fact Check ---
def keyword_fact_check(sentence):
    sentence_lower = sentence.lower()
    for phrase, label in KNOWN_FACTS:
        if phrase in sentence_lower:
            print(f"  KEYWORD OVERRIDE → {label} (matched: '{phrase}')")
            return label, 0.95
    return None, 0.0

# --- NLI Entailment Check ---
def check_entailment(sentence, context_chunks):
    # Layer 1: keyword check
    kw_label, kw_score = keyword_fact_check(sentence)
    if kw_label:
        return kw_label, kw_score

    # Layer 2: NLI check
    combined_context = " ".join(context_chunks)
    words = combined_context.split()
    if len(words) > 400:
        combined_context = " ".join(words[:400])

    results = nli_pipeline(
        combined_context + " [SEP] " + sentence,
        truncation=True,
        top_k=None,
        max_length=512
    )

    if isinstance(results[0], list):
        results = results[0]

    scores = {r['label'].lower(): r['score'] for r in results}
    print(f"  NLI scores → {scores}")

    entail_score = scores.get('entailment', 0)
    neutral_score = scores.get('neutral', 0)
    contra_score  = scores.get('contradiction', 0)

    if entail_score > neutral_score and entail_score > contra_score:
        return 'entailment', entail_score
    elif contra_score > neutral_score and contra_score > entail_score:
        return 'contradiction', contra_score
    else:
        return 'neutral', neutral_score

# --- Semantic Similarity ---
def semantic_similarity_score(answer, context_chunks):
    combined = " ".join(context_chunks)
    ans_emb = embed_model.encode(answer, convert_to_tensor=True)
    ctx_emb = embed_model.encode(combined, convert_to_tensor=True)
    return util.cos_sim(ans_emb, ctx_emb).item()

# --- Hallucination Score ---
def compute_hallucination_score(answer, context_chunks):
    sentences = split_sentences(answer)
    total_score = 0
    sentence_results = []

    for sent in sentences:
        label, score = check_entailment(sent, context_chunks)
        sentence_results.append({"sentence": sent, "label": label, "score": round(score, 2)})

        if label == "entailment":
            total_score += 1.0
        elif label == "neutral":
            total_score += 0.2 if score > 0.80 else 0.5
        else:
            total_score += 0.0

    support_ratio = total_score / len(sentences) if sentences else 0
    sem_score = semantic_similarity_score(answer, context_chunks)
    hallucination_score = 1 - (0.6 * support_ratio + 0.4 * sem_score)
    return round(hallucination_score, 2), sentence_results

# --- Classifier ---
def classify_hallucination(score):
    if score < 0.3:
        return "✅ Supported"
    elif score < 0.65:
        return "⚠️ Partially Supported"
    else:
        return "🔴 Hallucinated"

# --- Main Module ---
def hallucination_detection_module(query, answer, context_chunks):
    answer = clean_text(answer)
    score, sentence_details = compute_hallucination_score(answer, context_chunks)
    label = classify_hallucination(score)

    print(f"\n🔍 Query: {query}")
    print(f"📄 Answer: {answer}")
    print(f"\n📊 Hallucination Score: {score}")
    print(f"🏷️ Classification: {label}")
    print("\n📝 Sentence-level Analysis:")
    for item in sentence_details:
        print(f"  [{item['label'].upper()}] ({item['score']}) {item['sentence']}")

    return {
        "query": query,
        "answer": answer,
        "hallucination_score": score,
        "label": label,
        "sentence_analysis": sentence_details,
        "pass_to_next": score < 0.65
    }

print("✅ All functions loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

✅ All functions loaded successfully!


In [ ]:
label, score = keyword_fact_check("It is caused by eating too much sugar as a child.")
print(label, score)
# Should print: contradiction 0.95

  KEYWORD OVERRIDE → contradiction (matched: 'caused by eating too much sugar')
contradiction 0.95


In [ ]:
query = "What is diabetes?"
context = get_context(query, k=5)
answer = "Diabetes affects blood sugar. It is caused by eating too much sugar as a child. It doesnt spikes insulin and scary for our life."
result = hallucination_detection_module(query, answer, context)

  NLI scores → {'entailment': 0.8568804860115051, 'neutral': 0.1394057273864746, 'contradiction': 0.0037137989420443773}
  KEYWORD OVERRIDE → contradiction (matched: 'caused by eating too much sugar')
  KEYWORD OVERRIDE → contradiction (matched: 'doesnt spikes insulin')

🔍 Query: What is diabetes?
📄 Answer: Diabetes affects blood sugar. It is caused by eating too much sugar as a child. It doesnt spikes insulin and scary for our life.

📊 Hallucination Score: 0.54
🏷️ Classification: ⚠️ Partially Supported

📝 Sentence-level Analysis:
  [ENTAILMENT] (0.86) Diabetes affects blood sugar.
  [CONTRADICTION] (0.95) It is caused by eating too much sugar as a child.
  [CONTRADICTION] (0.95) It doesnt spikes insulin and scary for our life.
